[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/AcademiXBase/deep-learning-from-scratch/blob/dev/my_notebooks/ch05_02.ipynb)

In [ ]:
# Colab で実行している場合、リポジトリをクローンする
#!git clone -b dev https://github.com/AcademiXBase/deep-learning-from-scratch.git
#%cd deep-learning-from-scratch/my_notebooks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 先週の内容

### 1. 誤差逆伝播法

誤差逆伝播法 (Backpropagation) とは、ニューラルネットワークの重みパラメータの勾配を効率良く計算するための手法で、これは「合成関数の微分」と「連鎖律」にもとづいている  

#### ● 合成関数の微分としてのニューラルネットワーク  

ニューラルネットワークは、多くの層 (関数) が重なった巨大な合成関数と見なすことができ、例えば、$z=(x+y)^2$ という計算は、以下の 2 つの式から構成される合成関数である

$z = t^2$  

$t = x + y$  

#### ● 連鎖律 (Chain Rule) の原理  

連鎖律とは、「合成関数の微分がそれを構成する各関数の微分の積 (掛け算) として表すことができる」という性質のことである。前述の例について、$x$ に関する $z$ の微分 ($\dfrac{\partial z}{\partial x}$) を求めたいとき、次のように計算できる。  

$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial t} \times \frac{\partial t}{\partial x}$$

これは「局所的な微分」(それぞれの小さな計算の微分) を順次掛け合わせていけば、最終的には関数全体の微分が求められるということを表す。  

#### ● 誤差逆伝播法による解析的な微分  

微分には、微小な差分から近似的に微分値を求める「数値微分」と、数式の展開によって真の微分を求める「解析微分」の 2 種類がある。  

誤差逆伝播法とは、計算グラフ (5 章の適当な図を参照) の上でこの連鎖律を右 (出力層側) から左 (入力層側) へと適用していくことで、解析的に微分を効率良く求める手法である。  

* 順伝播 (Forward)：通常の計算を左から右へ進める。  

* 逆伝播 (Backward)：右から流れてきた微分値に対して、そのノードの「局所的な微分」を乗算して左へ流す。  

このように、複雑な数式を一つ一つの単純な計算 (ノード) に分解し、連鎖律に従って微分の値を伝播させることで、層が深いネットワークであっても、各パラメータの勾配を正確かつ高速に算出することが可能になる。  

### 2. 乗算レイヤ (MulLayer)  

* 順伝播 (Forward)：入力された 2 つの値を掛け合わせて出力する。  

    x と y の 2 つの入力の積をとり、1 つの出力を返す。  

* 逆伝播 (Backward)：上流 (出力層側) から伝わってきた微分値に対して、順伝播時の入力を「入れ替えた値」を乗算して下流 (入力層側) へ流す。  

    出力層側からの逆伝播として値を 1 つ受け取り、それを x と y それぞれで偏微分した値として入力層側に返す。  

    例えば、$x$ に関する偏微分を求める際は「上流の微分 $\times \, y$」、yに関する微分を求める際は「上流の微分 $\times \, x$」となる。  

* 実装のポイント：逆伝播の計算に順伝播時の入力値が必要になるため、インスタンス変数として入力を保持しておく。  

### 3. 加算レイヤ (AddLayer)  

* 順伝播 (Forward)：入力された 2 つの値を足し合わせて出力する (例：$x+y$)。

* 逆伝播 (Backward)：上流 (出力層側) から伝わってきた微分値を、そのまま下流 (入力層側) へ流す。  

    これは加算ノードでは局所的な微分が 1 となるため、上流の値に 1 を掛けても値が変わらないから。  

* 実装のポイント：乗算レイヤとは異なり、逆伝播の計算に順伝播時の入力値を使用しないため、入力を保持しておく必要はない。  

## 5.5 活性化関数レイヤの実装

### 5.5.1 ReLU レイヤ

ReLU (Rectified Linear Unit) は次のように定義され、  

$$ \text{ReLU}(x) = \max{(0,x)}$$

$x > 0$ のときには $x$ を返し、$ x \le 0$ のときには 0 を返す。  

これは次のように書くこともでき、  

$$
y = 
\begin{cases}
x & (x > 0)\\
0 & (x \leq 0)
\end{cases}
\tag{5.7}
$$

その $x$ に関する微分 (逆伝播) は、順伝播時の入力 $x$ の値により、  

$$
\frac{\partial y}{\partial x} =
\begin{cases}
1 & (x > 0)\\
0 & (x \leq 0)
\end{cases}
\tag{5.8}
$$

のように求まる。  

実装時には、入力された NumPy 配列の要素が「0 以下であるかどうか」を判定する mask という変数を使用する。  

forward (順伝播)：0 以下の要素を保持し、その場所の値を 0 に変換して出力する。  

backward (逆伝播)：順伝播時に保持した mask を使い、0 以下だった要素の微分値を 0 にして下流へ返す。  




In [ ]:
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout

        return dx

In [ ]:
#relu = Relu()

# x の範囲
x = np.linspace(-6, 6, 100)

# ReLU の出力 (forward)
#y = relu.forward(x)
y = np.maximum(0, x)

# ReLU の勾配 (backward)
#dx = relu.backward(np.ones_like(x))
dx = np.zeros_like(x)
dx[x > 0] = 1

# プロット
plt.figure(figsize=(7, 5))

plt.plot(x, y, color="C0", label="forward", linewidth=2)
plt.plot(x[x<0], dx[x<0], color="C1", label="backward")
plt.plot(x[x>0], dx[x>0], color="C1")
plt.scatter(0, 0, color="C1", s=50, zorder=3)  # 黒丸 (x ≤ 0)
plt.scatter(0, 1, facecolors="white", edgecolors="C1", s=50, zorder=3)  # 白抜き (x > 0)

# グラフの装飾
plt.title("ReLU")
plt.xlabel("x")
plt.ylabel("y, dy/dx")
plt.legend(fancybox=False, edgecolor="black", framealpha=1)
plt.grid(True)
plt.xlim(-6, 6)
plt.ylim(-1, 6)
plt.show()


### 5.5.2 Sigmoid レイヤ

Sigmoid 関数は次のように定義され、  

$$ y = \frac{1}{1 + e^{-x}} $$

ロジスティック (logistic) 関数とも呼ばれる。この関数は入力 $x$ の値が正で、かつ大きな値になるほど 1 に近づき、負の場合により小さくなるほど 0 に近づく。  

Sigmoid 関数を $y$ としたとき、その $x$ による微分は Sigmoid 関数自身を用いて

$$
\frac{dy}{dx} = y \, (1 - y)
$$

のように表すことができる。  

これは次のように計算した。  

$$
y = \frac{1}{1 + e^{-x}}
= (1 + e^{-x})^{-1}
$$

とすると、  

$$
\begin{aligned}
y &= t^{-1} \\
t &= 1 + e^{-x}
\end{aligned}
$$

といった合成関数の形に書き直せる (教科書の計算グラフではもっと細かく分割している)。  

それぞれを微分した式は、  

$$
\begin{aligned}
\frac{dy}{dt} &= -\,t^{-2} \\
\frac{dt}{dx} &= -\,e^{-x}
\end{aligned}
$$

この結果を使えば、  

$$
\frac{dy}{dt} \cdot \frac{dt}{dx}
= (-\,t^{-2}) \cdot (-\,e^{-x})
= \frac{e^{-x}}{t^{2}}
$$

となり、ここに $t = 1 + e^{-x}$ を代入すれば、  

$$
\begin{aligned}
\frac{dy}{dx}
&= \frac{e^{-x}}{(1 + e^{-x})^{2}} \\
&= \frac{1}{(1 + e^{-x})^{2}} \times e^{-x} \\
&= y^2 \, e^{-x}
\end{aligned}
$$

が求まる。  

これはまた、次のように変形することで、

$$
\begin{aligned}
y^2 \, e^{-x}
&= \frac{1}{(1+e^{-x})^{2}} \times e^{-x} \\[1em]
&= \frac{1}{1+e^{-x}} \, \frac{e^{-x}}{1+e^{-x}} \\[1em]
&= \left( \frac{1}{1+e^{-x}} \right) \left( \frac{1+e^{-x}}{1+e^{-x}} - \frac{1}{1+e^{-x}} \right) \\[1em]
&= \left( \frac{1}{1+e^{-x}} \right) \left( 1 - \frac{1}{1+e^{-x}} \right) \\[1em]
&= y \, (1-y)
\end{aligned}
$$

が最終的に得られる。  

結局、Sigmoid 関数の解析微分は、Sigmoid 関数 ($y = (1 + e^{-x})^{-1}$) 自身を用いて  

$$
\begin{aligned}
\frac{dy}{dx} = y \, (1-y)
\end{aligned}
$$

と表される。  

In [ ]:
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        out = 1 / (1 + np.exp(-x))
        self.out = out
        return out

    def backward(self, dout):
        dx = dout * self.out * (1 - self.out)
        return dx

In [ ]:
#sigmoid = Sigmoid()

# x の範囲
x = np.linspace(-6, 6, 100)

# Sigmoid の順伝播（出力値）
#y = sigmoid.forward(x)
y = 1 / (1 + np.exp(-x))

# Sigmoid の微分（勾配）
#dx = sigmoid.backward(np.ones_like(x))
dx = y * (1 - y)

# プロット
plt.figure(figsize=(7, 5))

plt.plot(x, y, label="forward", linewidth=2)
plt.plot(x, dx, label="backward", linewidth=2)

# グラフの装飾
plt.title("Sigmoid")
plt.xlabel("x")
plt.ylabel("y, dy/dx")
plt.grid(True)
plt.legend(fancybox=False, edgecolor="black", framealpha=1, loc="upper left")
plt.xlim(-6, 6)
plt.ylim(-0.5, 1.5)

plt.show()


## 5.6 Affine レイヤの実装

ニューラルネットワークの順伝播では、重み付き信号の総和を計算するために、行列の積 (ドット積) とバイアスの加算を用いる。これは幾何学の分野で「アフィン変換」と呼ばれるため、この処理を行う層を本書では Affine レイヤという名前で実装する。  

#### 1. 順伝播 (Forward Propagation)  

順伝播の計算は、入力 $X$、重み $W$、バイアス $B$ を用いて次のように表される。  

$$Y(1,3) = X(1,2) \cdot W(2,3) + B(1,3)$$

行列の積を計算する際には、対応する次元の要素数を一致させる必要がある  

#### 2. 逆伝播 (Backward Propagation)

行列を対象とした逆伝播もスカラー値の計算グラフと同様の手順で考えることができ、  

$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot W^T$$

$$\frac{\partial L}{\partial W} = X^T \cdot \frac{\partial L}{\partial Y}$$

となる。  

ここで、逆伝播においては、変数とその微分 (勾配) の形状は一致する。例えば、 $X$ の形状が (2,) であれば、その勾配 $\dfrac{\partial L}{\partial X}$ も同じ (2,) という形状になる。  

#### 3. バッチ版 Affine レイヤ  

実用的な学習では、$N$ 個のデータをまとめて処理するバッチ処理を行う。  

* 入力の形状：$N$ 個のデータを扱う場合、入力 $X$ の形状は (N,2) のようになる。  

* バイアスの逆伝播：順伝播では、バイアスは $N$ 個のデータのそれぞれの結果に対して加算される。そのため、逆伝播の際には各データの微分値を合算してバイアスの勾配を求める必要がある。実装上は `np.sum(dout, axis=0)` のように、データの軸 (第 0 軸) に対して総和を計算する。  

#### 4. Pythonによる実装  

Affine クラスとして実装する際は、初期化時に重み $W$ とバイアス $b$ を保持し、逆伝播時に各勾配 $dW$, $db$ をインスタンス変数として格納する。  


In [ ]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b

        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)

        return dx

### 5.6.3 Softmax-with-Loss レイヤ

Softmax-with-Loss レイヤとは、出力の正規化を行うソフトマックス関数と損失関数である交差エントロピー誤差を組み合わせたものを指す。  

#### 1. レイヤの役割と構造  

このレイヤは、学習フェーズにおいて最後に配置される。  

* ソフトマックス (Softmax) 関数：前の層 (Affine レイヤなど) から渡された値を入力として受け取り、それらを合計が 1 となるように割合 (確率) へと変換する。  

* 交差エントロピー誤差 (Cross Entropy Error)：正規化された確率と、正解ラベル (教師データ) を比較し、その誤差をひとつの値 (損失) として出力する。  

#### 2. 順伝播  

入力信号を $(a_1, \, a_2, \, a_3)$、正解ラベルを $(t_1, \, t_2, \, t_3)$ とすると、レイヤ全体で損失 $L$ を計算する。ただし、推論フェーズでは、一般的に Softmax レイヤは使用せず、直前の Affine レイヤの出力の最大値だけで判断するのが一般的。  

#### 3. 逆伝播  

このレイヤの最大の特徴は、逆伝播の結果が非常にシンプルになる点にある。  

予測された確率を $y$、正解ラベルを $t$ とすると、この Softmax-with-Loss レイヤからの逆伝播時の出力が $(y_1-t_1, \, y_2-t_2, \, y_3-t_3)$ のようにシンプルになる。これは現在の予測と正解の差 (誤差) そのものになっている。  


### 5.7.2 誤差逆伝播法に対応したニューラルネットワークの実装

前章 (第 4 章) の数値微分版からの主な変更点は、各処理を「レイヤ」としてモジュール化していること。これにより、予測や勾配計算をレイヤの伝播だけで簡潔に記述できるようになっている。  

class TwoLayerNet のインスタンス変数  

| 変数名       | 説明                                         |
| ----------- | -------------------------------------------- |
| `params`    | ニューラルネットワークのパラメータ (重み `W`、バイアス `b`) を保持するディクショナリ変数  |
| `layers`    | 各レイヤを保持する順番付きディクショナリ (`OrderedDict`)  |
| `lastLayer` | ネットワークの最後のレイヤ (この実装では SoftmaxWithLoss レイヤ)  |

TwoLayerNetクラスで保持される主要な変数は以下の通りである。  


class TwoLayerNet のメソッド  

| メソッド名            | 説明                                                             |
| -------------------- | ---------------------------------------------------------------- |
| `__init__`           | 初期化を行う。入力サイズ、隠れ層サイズ、出力サイズを指定し、重みの初期化とレイヤの生成 (Affine や Relu など) を行う |
| `predict`            | 推論 (認識) を行う。`layers` に格納された各レイヤの `forward` メソッドを順番に呼び出す |
| `loss`               | 損失関数の値を求める。`predict` の結果を `lastLayer` (Softmax-with-Loss) に渡して計算する |
| `accuracy`           | 認識精度を求める                                                  |
| `numerical_gradient` | 数値微分によって重みパラメータに対する勾配を求める (実装の検証用)     |
| `gradient`           | 誤差逆伝播法によって勾配を求める。数値微分に比べて圧倒的に高速        |

各機能は、前章の数値微分用のインタフェースを保持しつつ、内部でレイヤを活用する形に改良されている。  


In [ ]:
import sys, os
sys.path.append(os.pardir)  # 親ディレクトリのファイルをインポートするための設定
import numpy as np
from common.layers import *
from common.gradient import numerical_gradient
from collections import OrderedDict

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std = 0.01):
        # 重みの初期化
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

        # レイヤの生成
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])

        self.lastLayer = SoftmaxWithLoss()

    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)

        return x

    # x:入力データ, t:教師データ
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)

    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)

        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy

    # x:入力データ, t:教師データ
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)

        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])

        return grads

    def gradient(self, x, t):
        # forward
        self.loss(x, t)

        # backward
        dout = 1
        dout = self.lastLayer.backward(dout)

        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 設定
        grads = {}
        grads['W1'], grads['b1'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W2'], grads['b2'] = self.layers['Affine2'].dW, self.layers['Affine2'].db

        return grads

実装の重要ポイント：OrderedDict の活用  

レイヤは OrderedDict (順番付きディクショナリ) で保持している。  

これにより...   

* 順伝播 (Forward)：追加した順に各レイヤの forward() メソッドを呼ぶだけで完了する。  

* 逆伝播 (Backward)：レイヤを逆順に並び替え、各レイヤの backward() メソッドを呼ぶことで、連鎖律に従った勾配計算が自動的に行われる。  

逆伝播処理のロジックを抜粋すると、以下のように簡潔に表せる。  

```python
# 逆伝播（Backward）のフロー
dout = 1
dout = self.lastLayer.backward(dout)

layers = list(self.layers.values())
layers.reverse()  # レイヤを逆順にする
for layer in layers:
    dout = layer.backward(dout)  # 各レイヤの逆伝播を連続して呼び出す
```

このように、ネットワークの構成要素を「レイヤ」として独立させることで、層の追加や変更が柔軟に行えるようになる。  

数値微分と誤差逆伝播法の比較 (検証用)  

誤差逆伝播法は高速で正確だが、実装が複雑になりやすいためミスが起きる可能性がある。そのため、実装の正しさを確認するために、数値微分の結果と誤差逆伝播法の結果を比較する「勾配確認（gradient check）」を行うことが推奨されている。  

### 5.7.4 誤差逆伝播法を使った学習

機械学習では、損失から自動で計算されるパラメーターの他に手動であらかじめ設定する必要があるパラメーターも存在する。これをハイパーパラメーターと呼ぶ。  

`train_neuralnet.py` で使用される主な設定 (ハイパーパラメータ)  

| 変数名            | 説明                             | 設定例  |
| ---------------- | -------------------------------- | ------ |
| `iters_num`      | 勾配法による更新を繰り返す回数 (イテレーション数)           | 10,000 |
| `batch_size`     | 訓練データの中から一度に抽出するデータの数 (ミニバッチ)      | 100    |
| `learning_rate`  | 学習係数。勾配方向にどれだけパラメータを更新するかを決定     | 0.1    |
| `iter_per_epoch` | 1エポックあたりの繰り返し数 (全訓練データ数 ÷ バッチサイズ) | 600    |

In [ ]:
import sys, os
sys.path.append(os.pardir)

import numpy as np
from dataset.mnist import load_mnist

# データの読み込み
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]

    # 勾配
    #grad = network.numerical_gradient(x_batch, t_batch)
    grad = network.gradient(x_batch, t_batch)

    # 更新
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]

    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)

    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)


学習のメインループの処理フロー  

スクリプト内の `for` 文の中では、以下の 4 つのステップが繰り返し実行される。  

1. ミニバッチの取得：`np.random.choice()` を使い、訓練データ全体 (60,000 件) からランダムに `batch_size` 分のデータを取り出す。  

2. 勾配の算出：誤差逆伝播法 (`gradient()`) を用いて、現在のパラメータに対する損失関数の勾配を求める。  

3. パラメータの更新：算出された勾配に学習係数を掛けた値で、重み (`W`) とバイアス (`b`) を更新する。  

4. 途中経過の記録：各更新ステップでの損失 (Loss) の値を配列に保存し、学習の進行を確認できるようにする。  

評価の仕組み：エポックと認識精度  

ここでは、単に損失を減らすだけでなく、モデルが「まだ見ぬデータ (テストデータ)」に対して正しく反応できるか (汎化能力) を確認するため、1 エポックごとに認識精度を計算している。  

#### ● エポック (Epoch)：  

訓練データをすべて使い切った回数に対応する単位 (例：60,000 枚のデータを 100 枚ずつのバッチで処理する場合、600 回繰り返すと 1 エポックとなる)。  

#### ● 過学習 (Overfitting) の確認：  

訓練データの精度とテストデータの精度を比較し、両方が同じように向上していれば過学習が起きていないと判断できる。  